# Output Logit Analysis

Detailed analysis of the final logit distribution: shape, temperature sensitivity,
margin, rank distribution, and cross-position consistency.

## Why This Matters

Logit output analyzes how the model's output predictions form and change across layers. Understanding logit formation — which components contribute what to the final prediction — is central to explaining model behavior.

**Key references:**
- [nostalgebraist (2020) "interpreting GPT: the logit lens"](https://www.lesswrong.com/posts/AcKRB8wDpdaN6v6ru/interpreting-gpt-the-logit-lens) — Project intermediate residuals through the unembedding
- [Belrose et al. (2023) "Eliciting Latent Predictions"](https://arxiv.org/abs/2303.08112) — Learned affine probes improve on the logit lens

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.output_logit_analysis import (
    logit_distribution_profile, logit_temperature_sensitivity,
    logit_margin_analysis, logit_rank_distribution,
    cross_position_logit_consistency,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Logit Distribution Profile

Analyze the shape of the output logit distribution: mean, std, skewness,
entropy, and top/bottom tokens.

In [ ]:
result = logit_distribution_profile(model, tokens, position=-1)
print(f"Position {result['position']}:")
print(f"  Mean: {result['mean_logit']:.4f}, Std: {result['std_logit']:.4f}")
print(f"  Skewness: {result['skewness']:.4f}, Entropy: {result['entropy']:.4f}\n")
print('Top 5 tokens:')
for t in result['top_tokens']:
    print(f"  Token {t['token']:3d}: logit={t['logit']:+.4f}, prob={t['probability']:.4f}")
print('\nBottom 5 tokens:')
for t in result['bottom_tokens']:
    print(f"  Token {t['token']:3d}: logit={t['logit']:+.4f}, prob={t['probability']:.6f}")

## Temperature Sensitivity

How robust is the prediction to temperature scaling?

In [ ]:
result = logit_temperature_sensitivity(model, tokens)
robust = 'ROBUST' if result['is_robust'] else 'sensitive'
print(f"Base prediction: token {result['base_prediction']} [{robust}]")
print(f"Stable across {result['n_stable_temperatures']}/5 temperatures\n")
for entry in result['per_temperature']:
    same = '✓' if entry['same_prediction'] else '✗'
    print(f"  T={entry['temperature']:3.1f}: top={entry['top_token']:3d}, "
          f"conf={entry['confidence']:.4f}, entropy={entry['entropy']:.4f} [{same}]")

## Logit Margin Analysis

Margin between top-1 and top-2 predictions at each position.
Large margin = confident; small margin = uncertain.

In [ ]:
result = logit_margin_analysis(model, tokens)
print(f"Mean logit margin: {result['mean_logit_margin']:.4f}")
print(f"Decisive positions: {result['n_decisive']}/{len(result['per_position'])}\n")
for p in result['per_position']:
    bar = '█' * int(min(p['logit_margin'], 5) * 4)
    dec = ' [DECISIVE]' if p['is_decisive'] else ''
    print(f"  Pos {p['position']}: top1={p['top1_token']:3d} "
          f"({p['top1_logit']:+.3f} vs {p['top2_logit']:+.3f}), "
          f"margin={p['logit_margin']:.3f} {bar}{dec}")

## Logit Rank Distribution

How concentrated are the logits? Do a few tokens dominate,
or is probability spread across many?

In [ ]:
result = logit_rank_distribution(model, tokens)
conc = 'CONCENTRATED' if result['is_concentrated'] else 'spread'
print(f"Position {result['position']} [{conc}]")
print(f"  Top-1 prob: {result['top_1_probability']:.4f}")
print(f"  Top-5 prob: {result['top_5_probability']:.4f}")
print(f"  Top-10 prob: {result['top_10_probability']:.4f}")
print(f"  Tokens for 50%: {result['tokens_for_50_pct']}")
print(f"  Tokens for 90%: {result['tokens_for_90_pct']}")
print(f"  Tokens for 95%: {result['tokens_for_95_pct']}")

## Cross-Position Logit Consistency

How consistent are logit distributions across positions?
Measures pairwise KL divergence.

In [ ]:
result = cross_position_logit_consistency(model, tokens)
cons = 'CONSISTENT' if result['is_consistent'] else 'inconsistent'
print(f"Mean pairwise KL: {result['mean_pairwise_kl']:.4f} [{cons}]")
print(f"Outliers: {result['n_outliers']}\n")
for p in result['per_position']:
    outlier = ' [OUTLIER]' if p['is_outlier'] else ''
    bar = '█' * int(min(p['mean_kl_to_others'], 5) * 4)
    print(f"  Pos {p['position']}: mean_kl={p['mean_kl_to_others']:.4f} {bar}{outlier}")